In [1]:
print("OK")

OK


In [2]:
%pwd

'D:\\MedicalQuery\\research'

In [3]:
import os 
os.chdir("../")

In [4]:
# The '-U' ensures a clean update, and '--no-cache-dir' saves disk space
!pip install -U langchain langchain-community pypdf --no-cache-dir

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 524.3 kB/s eta 0:00:04
   -------- ------------------------------- 0.5/2.5 MB 524.3 kB/s eta 0:00:04
   -------- ------------------------------- 0.5/2.5 MB 524.3 kB/s eta 0:00:04
   ------------ --------------------------- 0.8/2.5 MB 453.1 kB/s eta 0:00:04
   ------------ --------------------------- 0.8/2.5 MB 453.1 kB/s eta 0:00:04
   ------------ --------------------------- 0.8/2.5 MB 453.1 kB/s eta 0:00:04
   ------------ --------------------------- 0.8/2.5 MB 453.1 kB/s eta 0:00:04
   ------------ --------------------------- 0.8/2.5 MB 453.1 kB/s eta 0:00:04
   ---------------- -----------

In [5]:
!pip install langchain-community pypdf --no-cache-dir

In [6]:
import os
import sys

try:
    # New official location for text splitters
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    # Standard locations for loaders
    from langchain_community.document_loaders import PyPDFLoader
    
    print("Success! All LangChain components are now correctly imported.")

    # --- Load the Medical Book from D: Drive ---
    file_path = "D:/MedicalQuery/data/Medical_book.pdf"
    loader = PyPDFLoader(file_path)
    
    print(f"Loading {file_path}...")
    documents = loader.load()

    # --- Split the text for MediQuery.ai ---
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(documents)

    print(f"Successfully created {len(text_chunks)} chunks.")

except ModuleNotFoundError as e:
    print(f"Still missing a module: {e}")
    print("Try running the !pip install command above one more time.")
except Exception as e:
    print(f"An unexpected snag occurred: {e}")

Success! All LangChain components are now correctly imported.
Loading D:/MedicalQuery/data/Medical_book.pdf...
Successfully created 5859 chunks.


In [7]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [8]:
# Update the path to point to your D: drive folder
DATA_PATH = "D:/MedicalQuery/data/"

def load_pdf_files(data_dir):
    loader = DirectoryLoader(
        data_dir,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

try:
    # Pass the D: drive path here
    extracted_data = load_pdf_files(DATA_PATH)
    print(f"Success! Loaded {len(extracted_data)} pages from {DATA_PATH}")
except FileNotFoundError:
    print(f"Error: The folder '{DATA_PATH}' was not found. Please verify the folder name on your D: drive.")
except Exception as e:
    print(f"Snag: {e}")

Snag: name 'DirectoryLoader' is not defined


In [9]:
extracted_data

NameError: name 'extracted_data' is not defined

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
# This is the new home for the Document class
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs = []
    for doc in docs:
        # Keep only the essential source info to save space in Pinecone
        source = doc.metadata.get("source", "Unknown")
        minimal_docs.append(Document(page_content=doc.page_content, metadata={"source": source}))
    return minimal_docs

# Try running the filter
try:
    # Use the extracted_data from your previous cell
    filtered_docs = filter_to_minimal_docs(extracted_data)
    print(f"Success! Filtered {len(filtered_docs)} documents.")
except NameError:
    print("Error: 'extracted_data' not defined. Make sure you ran the loader cell first!")

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [10]:
minimal_docs

NameError: name 'minimal_docs' is not defined

In [11]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [12]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

NameError: name 'minimal_docs' is not defined

In [13]:
texts_chunk

NameError: name 'texts_chunk' is not defined

In [14]:
!pip install langchain-huggingface --no-cache-dir

In [15]:
# This installs the actual model engine and the langchain wrapper
!pip install sentence-transformers langchain-huggingface --no-cache-dir

In [16]:
import requests
import time

class CloudEmbeddings:
    def __init__(self, hf_token):
        self.url = "https://api-inference.huggingface.co/pipeline/feature-extraction/sentence-transformers/all-MiniLM-L6-v2"
        self.headers = {"Authorization": f"Bearer {hf_token}"}

    def embed_query(self, text):
        response = requests.post(self.url, headers=self.headers, json={"inputs": text})
        return response.json()

    def embed_documents(self, texts):
        # The API prefers lists for document embedding
        response = requests.post(self.url, headers=self.headers, json={"inputs": texts})
        # If the model is 'loading' on HF servers, we wait and retry
        if isinstance(response.json(), dict) and "estimated_time" in response.json():
            time.sleep(response.json()["estimated_time"])
            return self.embed_documents(texts)
        return response.json()

# --- USE YOUR TOKEN HERE ---
HF_TOKEN = "your_hf_token_here" 
embedding = CloudEmbeddings(HF_TOKEN)

# Test it - this should work instantly!
try:
    vector = embedding.embed_query("Final test for MediQuery")
    print(f"Success! Cloud Vector length: {len(vector)}")
except Exception as e:
    print(f"Cloud Snag: {e}")

Success! Cloud Vector length: 1


In [17]:
vector = embedding.embed_query("Hello world")
vector

{'error': 'https://api-inference.huggingface.co is no longer supported. Please use https://router.huggingface.co instead.'}

In [18]:
print( "Vector length:", len(vector))

Vector length: 1


In [19]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [20]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [21]:
!pip uninstall -y pinecone-client
!pip install pinecone

In [22]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [23]:
pc

In [24]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [25]:
import sys
!{sys.executable} -m pip install -U langchain-pinecone langchain

'C:\python' is not recognized as an internal or external command,
operable program or batch file.


In [26]:
import os
os.environ["PINECONE_API_KEY"] = "pcsk_QNQyy_Tm2mhz1xHGywVnrLy4zstfTBgmnc75GKmPV6sSPJ6sJFTL1FoRVB87Pi2QDz5Zf"

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [28]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Replace "data/" with the actual folder path where your medical PDFs are stored
loader = PyPDFDirectoryLoader("Medical_book")
data = loader.load()

print(f"Loaded {len(data)} pages from the directory.")

Loaded 0 pages from the directory.


In [29]:
!pip install --no-cache-dir sentence-transformers

In [ ]:
import sentence_transformers
print("Success! Sentence Transformers is installed.")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded into MediQuery.ai!")

In [ ]:
import os
from langchain_pinecone import PineconeVectorStore

# 1. Ensure the API Key is in the environment
os.environ["PINECONE_API_KEY"] = "pcsk_QNQyy_Tm2mhz1xHGywVnrLy4zstfTBgmnc75GKmPV6sSPJ6sJFTL1FoRVB87Pi2QDz5Zf"

# 2. Use the verified index name
index_name = "medical-chatbot"

# 3. Connect to the existing index
try:
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embeddings # Using the 'embeddings' variable we loaded earlier
    )
    print(f"✅ Success! MediQuery.ai is now connected to the '{index_name}' index.")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore

# 1. Load your medical book
# Double check the path—if it's in your project folder, "data/Medical_book.pdf" works too.
loader = PyPDFLoader("D:/MedicalQuery/data/Medical_book.pdf")
data = loader.load()

# 2. Split the text into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(data)

print(f"✅ Documents split into {len(docs)} chunks.")

# 3. Upload to Pinecone
# This uses the 'embeddings' object we verified earlier
index_name = "medical-chatbot"

docsearch = PineconeVectorStore.from_documents(
    docs, 
    embeddings, 
    index_name=index_name
)

print(f"🚀 Success! {len(docs)} chunks uploaded to '{index_name}'.")

In [ ]:
query = "What are the common symptoms of the condition?"
test_results = docsearch.similarity_search(query, k=3)

for i, doc in enumerate(test_results):
    print(f"Snippet {i+1}: {doc.page_content[:200]}...")

In [ ]:
query = "What are the common symptoms discussed in the documents?"

# Fetch the 3 most relevant snippets from your medical PDFs
docs = docsearch.similarity_search(query, k=3)

# Print the results
if not docs:
    print("Zero results found. This means the index exists but might be empty.")
else:
    for i, doc in enumerate(docs):
        print(f"--- Snippet {i+1} ---")
        print(doc.page_content[:400] + "...") 
        print("-" * 30)

# Add more data to the existing Pinecone index

In [ ]:
from langchain_core.documents import Document

# 1. Define your test document with your name!
salony = Document(
    page_content="salony is a youtube channel that provides tutorials on various topics.",
    metadata={"source": "Youtube"}
)

# 2. Add it to your existing Pinecone index
# Use the 'salony' variable here to match your definition above
docsearch.add_documents([salony])

print("✅ 'salony' test document successfully added to 'medical-chatbot'!")

In [ ]:
query = "Tell me about Salony's channel."
results = docsearch.similarity_search(query, k=1)

if results:
    print(f"Success! Found: {results[0].page_content}")
else:
    print("Not found yet. Give Pinecone a few more seconds to index.")

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [76]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='29b8a12c-9034-4487-a5e0-3fb0d7470c21', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 37, 'page_label': '38', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'D:/MedicalQuery/data/Medical_book.pdf', 'total_pages': 637}, page_content='Nancy J. Nordenson\nAcid reflux see Heartburn\nAcidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'),
 Document(id='3ddda97f-08ce-4140-8f5b-50212da4f224', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-0

In [77]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [82]:
!pip install langchain-classic

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 699.0 kB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 780.2 kB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 780.2 kB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 817.6 kB/s  0:00:01
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
 

In [84]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# The rest of your code remains the same!
# question_answer_chain = create_stuff_documents_chain(llm, prompt)
# rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 1. Initialize the LLM (Ensure your OPENAI_API_KEY is set in os.environ)
llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# 2. Set up the Retriever using your existing docsearch
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

# 3. Define the Prompt (This tells the bot how to behave)
system_prompt = (
    "You are a medical assistant for MediQuery.ai. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Create the final RAG Chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("🚀 MediQuery.ai is now ready to answer medical questions!")

🚀 MediQuery.ai is now ready to answer medical questions!


In [85]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [86]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [90]:
!pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [91]:
import os
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Set the Groq API Key
os.environ["GROQ_API_KEY"] = "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"

# 2. Initialize the Groq LLM (Llama-3 is excellent for medical summaries!)
llm = ChatGroq(model_name="llama3-8b-8192", temperature=0.2)

# 3. Use the system prompt we defined for your medical assistant
system_prompt = (
    "You are a professional medical assistant for MediQuery.ai. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Re-build the chain using your existing 'docsearch' retriever
retriever = docsearch.as_retriever(search_kwargs={"k": 3})
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ MediQuery.ai is now powered by Groq and ready!")

✅ MediQuery.ai is now powered by Groq and ready!


In [93]:
import os
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Set Key
os.environ["GROQ_API_KEY"] = "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"

# Init LLM
llm = ChatGroq(model_name="llama3-8b-8192", temperature=0.2)

# Re-build and Invoke
# (Ensure your 'docsearch' and 'prompt' are already defined above)
# response = rag_chain.invoke({"input": "What is Acromegaly and gigantism?"})

In [96]:
from langchain_groq import ChatGroq

# 1. Update to the newer, supported Llama 3.1 model
# 'llama-3.1-8b-instant' is fast and free for most project tiers
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1)

# 2. Re-build the chain (using the prompt and retriever we defined)
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ MediQuery.ai updated to Llama 3.1! Ready to test.")

✅ MediQuery.ai updated to Llama 3.1! Ready to test.


In [97]:
try:
    response = rag_chain.invoke({"input": "What is Acromegaly and gigantism?"})
    print("--- MediQuery.ai Answer ---")
    print(response["answer"])
except Exception as e:
    print(f"❌ Error: {e}")

--- MediQuery.ai Answer ---
Acromegaly and gigantism are two related disorders caused by an abnormal release of a chemical from the pituitary gland in the brain. 

Acromegaly is a disorder in which the abnormal release of this chemical causes increased growth in bone and soft tissue, as well as various other disturbances throughout the body. This condition typically occurs in adults after normal growth has stopped, and it can lead to a range of symptoms, including enlarged hands and feet, joint pain, and vision problems.

Gigantism, on the other hand, is a similar condition that occurs in children before their bones have stopped growing. In this case, the abnormal release of the chemical from the pituitary gland causes excessive growth, leading to an unusually tall stature.


In [98]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.


In [99]:
response = rag_chain.invoke({"input": "what is the Treatment of Acne?"})
print(response["answer"])

The treatment of acne depends on the severity of the condition, which can be mild, moderate, or severe. 

For mild non-inflammatory acne, the treatment typically consists of reducing the formation of new comedones (blackheads and whiteheads) with topical medications such as:

1. Tretinoin: This is especially effective as it increases the turnover of skin cells.
2. Benzoyl peroxide: This helps to kill bacteria that can cause acne.
3. Adapalene: This is a retinoid that helps to prevent clogged pores.
4. Salicylic acid: This helps to exfoliate the skin and unclog pores.

When acne is complicated by inflammation, topical antibiotics may be added to the treatment regimen.

For more severe cases of acne, treatment may involve a combination of topical and oral medications, including:

1. Oral antibiotics: These can help to reduce inflammation and kill bacteria that cause acne.
2. Hormonal treatments: These may be prescribed for acne that is related to hormonal imbalances.
3. Isotretinoin: Thi

In [101]:
response = rag_chain.invoke({"input": "salony?"})
print(response["answer"])

I found a mention of the YouTube channel "salony" in the context you provided. However, there is no additional information about the channel, its content, or its purpose. It seems to be a brief mention in a passage about alopecia and thyroid disorders. If you're looking for information about the channel, I don't have any specific details to provide.
